# Verify 499 archived files — no full Google Drive scan

**Fixed the Google Drive 403 rate limit:** verification now uses the already-generated `transfer_results.csv` report rather than requesting all 6,799 Drive records again. It downloads Hugging Face copies for MD5 comparison against the source Drive checksums recorded during transfer. Only 8 native Google Docs require fresh Drive exports (with automatic retry/backoff).

**First run only:** select your existing `transfer_results (2).csv` when Colab opens the file picker. The notebook saves **only the 499 eligible file records** to the private Hugging Face audit repository. Subsequent runs reuse that private manifest automatically, and resume prior verified checks.

Run **Runtime → Run all**. The program does **not** upload or delete university source files, does not modify archived files, and stores only verification CSV reports in a private audit repo. Full byte verification may download approximately 9.67 GB once; partial checks are saved every 25 files.


In [ ]:
%pip -q install 'huggingface_hub==0.36.0' 'google-auth==2.49.0' 'google-api-python-client'


In [ ]:
"""Google Drive → private Hugging Face archives. Run in Google Colab.

Safety defaults:
- Audit only, not upload.
- Never remove or rename source files on Google Drive.
- Deduplicate only byte-identical Google Drive files with matching MD5 and size.
- Skip sensitive datasets/documents until separately authorized.
- Verify a remote LFS SHA256+size where metadata is available.
"""

from __future__ import annotations

import csv
import hashlib
import os
import re
import tempfile
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any

# 1. These are Drive IDs previously verified against the user's connected Drive.
# Each root gets a unique canonical source label. A Drive ID encountered twice
# is counted and uploaded once per project.
SOURCES = {
    "university-resnet50-pneumonia-archive": [
        ("academic-coursework", "1Ch62wticPBABtiaH7fCbIReTq1BXuVU4", "folder"),
        ("experiment-workspace", "1cQeYJ6VCOZc-zztC5TQa8AkZ-8rkfDbM", "folder"),
    ],
    "university-industrial-defects-polyloss-archive": [
        ("coursework-and-evidence", "1OI3zrAtmA9ZfwH3l_DB3IiwFBK2P1_ny", "folder"),
    ],
    "university-lab06-document-quality-archive": [
        ("lab06-experiments", "1ja5uljMIcMSNLd41scx2HHV5B0SLallu", "folder"),
        ("lab06-reports-and-tasks", "1id2pTjvu1SnNwshRJjRc-Teno5AGGBIg", "folder"),
        ("original-dataset-zip", "1QNHsbtpWcmeo0asG2Mrrruwy5w2lmobB", "file"),
    ],
    "university-summer-practice-2025-archive": [
        ("digit-recognition-project", "1LmOjZmJxwdVzN7xMAxLaJYQOd2HACjkn", "folder"),
    ],
    "university-summer-practice-2026-archive": [
        ("equipment-booking-project", "1u3kMCTVIab07B3mpLclCUOfFWEK7Wbk_", "folder"),
    ],
}

# User has explicitly authorized transfer of non-sensitive, distinct archive files.
# The actual upload still requires user-authenticated Google Drive and a HF write token.
SYNC_ENABLED = True
EXPECTED_HF_USERNAME = "Zkri2026"
# Source document photos and clinical datasets need a separate licensing/privacy decision.
ALLOW_SENSITIVE_DATA = False
# If True, CRC-check every downloaded .zip before upload (may be CPU-intensive).
TEST_ZIP_CONTENTS = True
# During the audit, validate whether split .zip.part001... files reproduce a full ZIP.
CHECK_SPLIT_ARCHIVES = True
# If true, require the Hugging Face LFS SHA256, not just file-size confirmation.
REQUIRE_REMOTE_SHA256 = True
# Optional subset, e.g. ['university-resnet50-pneumonia-archive']; None = all.
ONLY_PROJECTS = None

NATIVE_EXPORTS = {
    "application/vnd.google-apps.document": ("application/pdf", ".pdf"),
    "application/vnd.google-apps.spreadsheet": (
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": (
        "application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
}
ALLOWED_REPO_CHARS = re.compile(r"[^a-zA-Z0-9._-]+")


def safe_filename(s: str) -> str:
    s = s.strip().replace("\\", "_").replace("/", "_")
    s = re.sub(r"[\x00-\x1f]", "_", s)
    return s[:180] or "unnamed"


def file_category(path: str, name: str) -> str:
    lower = ("/" + path + "/" + name).lower()
    ext = Path(name).suffix.lower()
    if ext in (".pt", ".pth", ".ckpt", ".keras", ".onnx", ".h5", ".safetensors"):
        return "models"
    if any(x in lower for x in ("/dataset", "/raw/", "/images/", "/data/")) or \
            "dataset" in lower or ext in (".parquet", ".tfrecord"):
        return "datasets"
    if ext in (".py", ".ipynb", ".sh", ".ps1", ".cpp", ".h", ".ts", ".tsx", ".kt", ".java"):
        return "code"
    if any(x in lower for x in ("/results", "/outputs", "/experiments", "checkpoint")) \
            or ext in (".json", ".jsonl", ".csv", ".png", ".jpg"):
        return "results"
    if ext in (".pdf", ".doc", ".docx", ".pptx", ".md", ".txt", ".xlsx"):
        return "documents"
    return "other"


def sensitivity(project: str, path: str, name: str) -> str:
    lower = (path + "/" + name).lower()
    base = name.lower()
    # Secrets always take precedence, including when located inside a dataset folder.
    if base in ("kaggle.json", ".env", "secrets.json", "credentials.json") or \
            base.startswith(".env.") or base.endswith((".pem", ".key", ".p12", ".pfx", ".keystore")):
        return "NEVER_UPLOAD_SECRET"
    # Source images from passport, SNILS, INN may contain identifying information,
    # even when synthetically augmented. The entire Lab06 dataset is gated.
    if project == "university-lab06-document-quality-archive":
        if any(k in lower for k in ("dataset", "passport", "snils", "inn", "raw/", "yolo_data")):
            return "REVIEW_PERSONAL_DOCUMENT_IMAGES"
    if project == "university-resnet50-pneumonia-archive":
        if any(k in lower for k in ("/dataset", "/data/", "chest_xray", "patient", "clinical")):
            return "REVIEW_MEDICAL_DATA_LICENSE"
    return ""


def sha256_file(filepath: Path) -> str:
    h = hashlib.sha256()
    with filepath.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def connect_drive():
    try:
        from google.colab import auth  # type: ignore
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build
    except ImportError as e:
        raise RuntimeError("Run this script in Google Colab with your Google account.") from e
    creds, _ = google.auth.default()
    return build("drive", "v3", credentials=creds, cache_discovery=False)


def drive_meta(service, file_id: str) -> dict[str, Any]:
    return service.files().get(fileId=file_id, fields=(
        "id,name,mimeType,size,md5Checksum,parents,webViewLink,description,createdTime,modifiedTime,trashed"
    ), supportsAllDrives=True).execute()


def direct_children(service, folder_id: str):
    page = None
    while True:
        r = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum,parents,webViewLink,description,createdTime,modifiedTime)",
            pageSize=1000,
            pageToken=page,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        yield from r.get("files", [])
        page = r.get("nextPageToken")
        if not page:
            break


def collect_files(service, project: str, roots):
    pending = []
    for label, file_id, kind in roots:
        if kind == "folder":
            pending.append((file_id, label, ""))
        else:
            f = drive_meta(service, file_id)
            yield project, label, f.get("name", file_id), f
    visited_folders = set()
    while pending:
        folder_id, label, relpath = pending.pop()
        if folder_id in visited_folders:
            continue
        visited_folders.add(folder_id)
        for f in direct_children(service, folder_id):
            name = safe_filename(f.get("name", f["id"]))
            child_path = f"{relpath}/{name}".strip("/")
            if f.get("mimeType") == "application/vnd.google-apps.folder":
                pending.append((f["id"], label, child_path))
            elif f.get("mimeType") == "application/vnd.google-apps.shortcut":
                # Shortcuts have no bytes of their own. Do not follow arbitrary targets.
                continue
            else:
                yield project, label, child_path, f


def target_path(label: str, original_path: str, mime: str) -> str:
    path = Path(original_path)
    name = safe_filename(path.name)
    suffix = NATIVE_EXPORTS.get(mime)
    if suffix:
        name += suffix[1]
    category = file_category(original_path, name)
    ancestors = [safe_filename(x) for x in path.parts[:-1]]
    return "/".join([category, safe_filename(label), *ancestors, name])


def plan(service):
    records = []
    for project, roots in SOURCES.items():
        if ONLY_PROJECTS and project not in ONLY_PROJECTS:
            continue
        hashes = {}
        ids = {}
        destinations = set()
        for _, label, path, f in collect_files(service, project, roots):
            fid = f["id"]
            size = int(f.get("size") or 0)
            md5 = f.get("md5Checksum") or ""
            mime = f.get("mimeType", "")
            dst = target_path(label, path, mime)
            danger = sensitivity(project, path, f.get("name", ""))
            status = "UPLOAD"
            duplicate_of = ""
            if danger == "NEVER_UPLOAD_SECRET":
                status = "SKIP_SECRET"
            elif danger and not ALLOW_SENSITIVE_DATA:
                status = "REVIEW_SENSITIVE"
            elif fid in ids:
                status, duplicate_of = "SKIP_SAME_FILE", ids[fid]
            elif md5 and (md5, size) in hashes:
                status, duplicate_of = "SKIP_EXACT_DUPLICATE", hashes[(md5, size)]
            if status == "UPLOAD":
                if dst in destinations:
                    suffix = (md5 or fid).replace("/", "")[:10]
                    dst = f"{dst}__{suffix}"
                destinations.add(dst)
                if md5:
                    hashes[(md5, size)] = dst
                ids[fid] = dst
            records.append({
                "project": project,
                "source_label": label,
                "source_path": path,
                "parent_id": (f.get("parents") or [""])[0],
                "file_id": fid,
                "mime_type": mime,
                "size_bytes": size,
                "drive_md5": md5,
                "destination_path": dst,
                "sensitivity_flag": danger,
                "plan": status,
                "duplicate_of": duplicate_of,
                "result": "NOT_STARTED",
                "sha256": "",
                "remote_sha256": "",
            })
    return records


def eliminate_verified_split_zip_copies(service, records):
    """Exact byte-level comparison: assembled .zip.partNNN vs complete .zip.

    Never assume matching names/sizes are sufficient. Download parts, stream
    their concatenation through MD5, compare against Drive's full ZIP MD5.
    Only if a verified whole file matches do we omit these redundant pieces.
    """
    if not CHECK_SPLIT_ARCHIVES:
        return
    groups = defaultdict(list)
    complete = {}
    for r in records:
        k = (r["project"], r["parent_id"], str(Path(r["source_path"]).parent))
        basename = Path(r["source_path"]).name
        match = re.match(r"^(.*\.zip)\.part(\d+)$", basename, flags=re.I)
        if match:
            groups[k + (match.group(1),)].append((int(match.group(2)), r))
        elif basename.lower().endswith(".zip"):
            complete[k + (basename,)] = r
    for key, parts in groups.items():
        full = complete.get(key)
        if not full or not full.get("drive_md5"):
            continue
        parts.sort(key=lambda x: x[0])
        numbers = [i for i, _ in parts]
        if numbers != list(range(1, len(numbers)+1)):
            print("SPLIT ARCHIVE INCOMPLETE:", key[-1], numbers)
            continue
        total = sum(r["size_bytes"] for _, r in parts)
        if total != full["size_bytes"]:
            print("SPLIT ARCHIVE SIZE MISMATCH:", key[-1])
            continue
        digest = hashlib.md5()
        try:
            for _, part in parts:
                with tempfile.TemporaryDirectory() as t:
                    f = Path(t) / "piece.bin"
                    download_drive_file(service, part, f)
                    with f.open("rb") as stream:
                        for block in iter(lambda: stream.read(8*1024*1024), b""):
                            digest.update(block)
        except Exception as e:
            print("Cannot verify split archive:", key[-1], e)
            continue
        if digest.hexdigest().lower() == full["drive_md5"].lower():
            print("VERIFIED REDUNDANT SPLIT ARCHIVE:", key[-1])
            for _, part in parts:
                if part["plan"] == "UPLOAD":
                    part["plan"] = "SKIP_VERIFIED_SPLIT_ARCHIVE"
                    part["duplicate_of"] = full["destination_path"]
        else:
            print("Split archive differs from complete ZIP; preserving all:", key[-1])


def write_csv(records, file_path: str):
    if not records:
        return
    with open(file_path, "w", newline="", encoding="utf-8-sig") as o:
        writer = csv.DictWriter(o, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)


def download_drive_file(service, record: dict, out: Path):
    from googleapiclient.http import MediaIoBaseDownload
    mime = record["mime_type"]
    if mime in NATIVE_EXPORTS:
        export_mime, _ = NATIVE_EXPORTS[mime]
        request = service.files().export_media(fileId=record["file_id"], mimeType=export_mime)
    else:
        request = service.files().get_media(fileId=record["file_id"], supportsAllDrives=True)
    with out.open("wb") as writer:
        loader = MediaIoBaseDownload(writer, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = loader.next_chunk(num_retries=4)


def verify_zip(path: Path):
    if path.suffix.lower() != ".zip":
        return
    with zipfile.ZipFile(path) as z:
        if TEST_ZIP_CONTENTS:
            corrupt = z.testzip()
            if corrupt:
                raise ValueError(f"ZIP CRC FAILED at {corrupt}")


def verify_remote(api, repo_id: str, target: str, expected_sha: str, expected_size: int):
    objs = api.get_paths_info(repo_id=repo_id, paths=[target], repo_type="dataset", expand=True)
    obj = objs[0] if objs else None
    if obj is None:
        raise ValueError("Remote path is missing after upload")
    if getattr(obj, "size", None) != expected_size:
        raise ValueError("Remote file size does not match local upload")
    lfs = getattr(obj, "lfs", None)
    remote_sha = lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
    if remote_sha:
        if remote_sha.lower() != expected_sha.lower():
            raise ValueError("Remote LFS SHA-256 does not match local file")
        return "VERIFIED_SHA256_AND_SIZE", remote_sha
    if REQUIRE_REMOTE_SHA256:
        return "UPLOADED_SIZE_ONLY_NOT_FULLY_VERIFIED", ""
    return "VERIFIED_SIZE_ONLY", ""




In [ ]:
"""Verify Hugging Face archive file bytes against Google Drive metadata/exports.

Read originals, download archived HF copies, compute hashes, checkpoint
progress to a separate private HF repo. No source files are modified/deleted.
"""

from __future__ import annotations
import csv
import hashlib
import tempfile
import time
import random
from collections import Counter
from pathlib import Path

OWNER = "Zkri2026"
AUDIT_REPO_NAME = "university-migration-integrity-audit"
AUDIT_FILE = "integrity_verification.csv"
MANIFEST_FILE = "migration_source_manifest_499.csv"
MANIFEST_FIELDS = ["project","file_id","source_path","destination_path","mime_type","size_bytes","drive_md5"]
EVERY = 25
COLUMNS = [
    "project","file_id","source_path","destination_path","mime_type",
    "source_size_bytes","source_drive_md5","hf_size_bytes",
    "hf_md5","hf_sha256","hf_lfs_sha256","source_export_sha256",
    "repo_revision","status","error"
]

def hashes(path):
    md5, sha, size = hashlib.md5(), hashlib.sha256(), 0
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):
            md5.update(chunk)
            sha.update(chunk)
            size += len(chunk)
    return md5.hexdigest(),sha.hexdigest(),size

def lfs_hash(meta):
    lfs=getattr(meta,"lfs",None)
    return (lfs.get("sha256","") if isinstance(lfs,dict)
            else (getattr(lfs,"sha256","") or ""))

def is_verified(status):
    return status.startswith(("VERIFIED_DRIVE_MD5_","VERIFIED_NATIVE_EXPORT_SHA256_",
                              "VERIFIED_SOURCE_SHA256_"))

def write_csv(rows):
    dest=Path(AUDIT_FILE)
    tmp=dest.with_suffix(".csv.partial")
    with tmp.open("w",encoding="utf-8-sig",newline="") as f:
        writer=csv.DictWriter(f,fieldnames=COLUMNS,extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    tmp.replace(dest)

def private_checkpoint(api,audit_repo,rows):
    write_csv(rows)
    api.upload_file(
        repo_id=audit_repo,repo_type="dataset",
        path_or_fileobj=AUDIT_FILE,path_in_repo=AUDIT_FILE,
        commit_message="Save migration integrity verification checkpoint"
    )

def get_previous(api,token,audit_repo):
    from huggingface_hub import hf_hub_download
    if AUDIT_FILE not in api.list_repo_files(audit_repo,repo_type="dataset"):
        return {}
    with tempfile.TemporaryDirectory(prefix="audit_resume_") as td:
        local=hf_hub_download(
            repo_id=audit_repo,repo_type="dataset",
            filename=AUDIT_FILE,token=token,cache_dir=td
        )
        with open(local,"r",encoding="utf-8-sig",newline="") as f:
            rows=list(csv.DictReader(f))
    return {(r["project"],r["file_id"],r["destination_path"]):r for r in rows}

def may_resume(old,record,revision):
    if not old or not is_verified(old.get("status","")):
        return False
    if record["mime_type"] in NATIVE_EXPORTS:
        return False  # re-export native Drive documents on every run
    return (
        old.get("repo_revision")==revision
        and old.get("source_drive_md5","").lower()==str(record.get("drive_md5") or "").lower()
        and old.get("source_size_bytes")==str(record.get("size_bytes") or 0)
    )

def verify_one(record,remote,revision,token,service):
    from huggingface_hub import hf_hub_download
    out={
        "project":record["project"],"file_id":record["file_id"],
        "source_path":record["source_path"],"destination_path":record["destination_path"],
        "mime_type":record["mime_type"],"source_size_bytes":record["size_bytes"],
        "source_drive_md5":record["drive_md5"],"hf_size_bytes":"",
        "hf_md5":"","hf_sha256":"","hf_lfs_sha256":"",
        "source_export_sha256":"","repo_revision":revision,
        "status":"PENDING","error":""
    }
    if remote is None:
        out["status"]="MISSING_HF_FILE"
        return out
    remote_size=getattr(remote,"size",None)
    out["hf_size_bytes"]=remote_size
    native=record["mime_type"] in NATIVE_EXPORTS
    if not native and remote_size!=int(record["size_bytes"]):
        out["status"]="MISMATCH_REMOTE_METADATA_SIZE"
        return out
    out["hf_lfs_sha256"]=lfs_hash(remote)
    with tempfile.TemporaryDirectory(prefix="hf_verify_") as td:
        remote_local=hf_hub_download(
            repo_id=f"{OWNER}/{record['project']}",
            filename=record["destination_path"],
            revision=revision,repo_type="dataset",
            token=token,cache_dir=td
        )
        md5,sha,size=hashes(remote_local)
        out["hf_md5"]=md5
        out["hf_sha256"]=sha
        out["hf_size_bytes"]=size
        if size!=remote_size:
            out["status"]="MISMATCH_HF_METADATA_VS_BYTES_SIZE"
            return out
        if out["hf_lfs_sha256"] and out["hf_lfs_sha256"].lower()!=sha.lower():
            out["status"]="MISMATCH_HF_LFS_SHA256"
            return out
        if native or not record["drive_md5"]:
            suffix=NATIVE_EXPORTS.get(record["mime_type"],(None,".bin"))[1]
            orig=Path(td)/("original"+suffix)
            drive_export_with_backoff(service,record,orig)
            _,orig_sha,orig_size=hashes(orig)
            out["source_export_sha256"]=orig_sha
            if orig_size!=size:
                out["status"]="MISMATCH_NATIVE_OR_SOURCE_SIZE"
            elif orig_sha!=sha:
                out["status"]="MISMATCH_NATIVE_OR_SOURCE_SHA256"
            elif native:
                out["status"]="VERIFIED_NATIVE_EXPORT_SHA256_AND_SIZE"
            else:
                out["status"]="VERIFIED_SOURCE_SHA256_AND_SIZE"
        elif md5.lower()!=record["drive_md5"].lower():
            out["status"]="MISMATCH_DRIVE_MD5"
        elif size!=int(record["size_bytes"]):
            out["status"]="MISMATCH_DRIVE_SIZE"
        elif out["hf_lfs_sha256"]:
            out["status"]="VERIFIED_DRIVE_MD5_HF_LFS_SHA256_AND_SIZE"
        else:
            out["status"]="VERIFIED_DRIVE_MD5_AND_SIZE"
    return out


def load_saved_transfer_manifest(api, token, audit_repo):
    """Load only the 499 migration candidates; never rescan Drive folders."""
    from huggingface_hub import hf_hub_download
    import os

    contents = api.list_repo_files(repo_id=audit_repo, repo_type="dataset")
    if MANIFEST_FILE in contents:
        local = hf_hub_download(
            repo_id=audit_repo, repo_type="dataset",
            filename=MANIFEST_FILE, token=token
        )
        print("Reusing previously saved private manifest. No Drive folder scan.")
    else:
        candidates = [
            Path("transfer_results.csv"), Path("transfer_results (2).csv"),
            Path("audit_manifest.csv")
        ]
        candidates += sorted(Path(".").glob("transfer_results*.csv"))
        found = next((x for x in candidates if x.exists()), None)
        if found:
            local = str(found)
            print("Using existing local migration report:",found.name)
        else:
            from google.colab import files
            print("ONE-TIME FILE SELECTION: choose transfer_results (2).csv")
            selected = files.upload()
            if not selected:
                raise RuntimeError("Please select the transfer_results.csv report.")
            local = next(iter(selected))
        with open(local, encoding="utf-8-sig", newline="") as source:
            reader = csv.DictReader(source)
            required = set(MANIFEST_FIELDS + ["plan"])
            if not required.issubset(reader.fieldnames or []):
                raise ValueError("Wrong CSV. Choose transfer_results.csv containing the 'plan' column.")
            candidates = [
                {k: r.get(k, "") for k in MANIFEST_FIELDS}
                for r in reader if r["plan"] == "UPLOAD"
            ]
        if len(candidates) != 499:
            raise ValueError(
                f"Expected 499 transfer candidates, found {len(candidates)}. "
                "Do not verify the wrong report."
            )
        if len({(r["project"],r["file_id"],r["destination_path"])
                for r in candidates}) != 499:
            raise ValueError("Duplicate entries detected in the transfer CSV.")
        for r in candidates:
            if not r["drive_md5"] and r["mime_type"] not in NATIVE_EXPORTS:
                raise ValueError(f"Missing source Drive MD5: {r['file_id']}")
        # Retain only 499 eligible paths in this PRIVATE Hugging Face audit repo.
        with tempfile.TemporaryDirectory(prefix="manifest_499_") as td:
            dest = Path(td)/MANIFEST_FILE
            with dest.open("w",encoding="utf-8-sig",newline="") as out:
                writer = csv.DictWriter(out,fieldnames=MANIFEST_FIELDS)
                writer.writeheader()
                writer.writerows(candidates)
            api.upload_file(
                repo_id=audit_repo, repo_type="dataset",
                path_or_fileobj=str(dest), path_in_repo=MANIFEST_FILE,
                commit_message="Store private transfer snapshot for 499 file integrity checks"
            )
        print("Saved 499 eligible source records privately. Never need to rescan 6799 files.")
        return candidates

    with open(local, encoding="utf-8-sig", newline="") as source:
        targets = list(csv.DictReader(source))
    if len(targets) != 499:
        raise ValueError(f"Private manifest has {len(targets)} rows; expected 499.")
    return targets


def drive_export_with_backoff(service, record, outfile):
    """Retry only temporary Drive rate-limit errors on 8 native document exports."""
    from googleapiclient.errors import HttpError
    for attempt in range(6):
        try:
            download_drive_file(service, record, outfile)
            return
        except HttpError as e:
            status = int(getattr(e.resp, "status", 0))
            details = str(e).lower()
            limited = status in (403, 429) and any(
                word in details for word in (
                    "ratelimitexceeded", "userratelimitexceeded",
                    "quota exceeded", "too many requests"
                )
            )
            if not limited or attempt == 5:
                raise
            delay = min(90, 3 * (2 ** attempt)) + random.uniform(0, 2)
            print(f"Drive temporarily rate-limited native export; retrying after {delay:.0f}s")
            time.sleep(delay)



def main_verify():
    from google.colab import userdata
    from huggingface_hub import HfApi

    token=userdata.get("HF_TOKEN")  # read ONCE
    if not token:
        raise RuntimeError("Add HF_TOKEN as a Colab Secret, and enable notebook access.")
    api=HfApi(token=token)
    username=api.whoami()["name"]
    if username.lower()!=OWNER.lower():
        raise RuntimeError(f"Connected to {username!r}; expected {OWNER!r}.")
    audit_repo=f"{username}/{AUDIT_REPO_NAME}"
    api.create_repo(repo_id=audit_repo,repo_type="dataset",private=True,exist_ok=True)
    if not api.repo_info(audit_repo,repo_type="dataset").private:
        raise RuntimeError("Audit repository is PUBLIC. Refusing to save private paths.")
    previous=get_previous(api,token,audit_repo)

    targets=load_saved_transfer_manifest(api,token,audit_repo)
    # Check 491 ordinary Drive binary files first. Postpone native Google
    # document exports until the end, giving the Drive quota time to recover.
    targets.sort(key=lambda r: r["mime_type"] in NATIVE_EXPORTS)
    print(f"Verifying {len(targets)} files from saved transfer snapshot.")
    print("No Google Drive folder listing. Only 8 native exports may access Drive.")
    needs_export=any(r["mime_type"] in NATIVE_EXPORTS for r in targets)
    service=connect_drive() if needs_export else None

    frozen,remote_tree={},{}
    for project in sorted({r["project"] for r in targets}):
        repo_id=f"{username}/{project}"
        info=api.repo_info(repo_id,repo_type="dataset")
        if not info.private:
            raise RuntimeError(f"Archive repository {repo_id} must be PRIVATE.")
        frozen[project]=info.sha
        remote_tree[project]={
            entry.path:entry
            for entry in api.list_repo_tree(
                repo_id,repo_type="dataset",recursive=True,
                expand=True,revision=info.sha
            )
            if getattr(entry,"size",None) is not None
        }
        print(project,":",len(remote_tree[project]),"remote files")

    rows=[]
    progress=0
    for number,record in enumerate(targets,1):
        key=(record["project"],record["file_id"],record["destination_path"])
        revision=frozen[record["project"]]
        old=previous.get(key)
        if may_resume(old,record,revision):
            rows.append(old)
            print(f"[{number}/{len(targets)}] CACHED VERIFIED:",record["source_path"])
            continue
        print(f"[{number}/{len(targets)}] VERIFY:",record["source_path"])
        try:
            result=verify_one(
                record,
                remote_tree[record["project"]].get(record["destination_path"]),
                revision,token,service
            )
        except Exception as e:
            result={
                "project":record["project"],"file_id":record["file_id"],
                "source_path":record["source_path"],
                "destination_path":record["destination_path"],
                "mime_type":record["mime_type"],
                "source_size_bytes":record["size_bytes"],
                "source_drive_md5":record["drive_md5"],
                "repo_revision":revision,
                "status":"ERROR_DOWNLOAD_OR_COMPARE","error":str(e)[:300]
            }
        print("  RESULT:",result["status"])
        rows.append(result)
        progress+=1
        write_csv(rows)
        if progress>=EVERY:
            try:
                private_checkpoint(api,audit_repo,rows)
                print("Checkpoint saved:",len(rows),"files")
                progress=0
            except Exception as e:
                print("Private checkpoint delayed; local CSV preserved:",str(e)[:160])
    try:
        private_checkpoint(api,audit_repo,rows)
    except Exception as e:
        print("Private audit upload failed; download local CSV:",str(e)[:170])
    print("VERIFICATION SUMMARY:")
    for status,count in sorted(Counter(r["status"] for r in rows).items()):
        print(status,count)
    print("Private audit: https://huggingface.co/datasets/"+audit_repo)
    print("Google Drive originals were NOT modified or deleted.")
    if len(rows)==499 and all(is_verified(r["status"]) for r in rows):
        print("ALL 499 FILES CONTENT VERIFIED. No deletion will occur automatically.")
    else:
        print("INCOMPLETE OR MISMATCHED: inspect CSV; DO NOT DELETE Drive files.")

main_verify()


In [ ]:
from pathlib import Path
from google.colab import files
if Path("integrity_verification.csv").exists():
    files.download("integrity_verification.csv")
else:
    print("No report yet; inspect the error in the previous cell.")
